## highz-accretion-atlas v1 evaluation

Build v1 science tables from `data/processed/v1_processed.csv` using the README growth assumptions.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from the current notebook or script directory."""
    current = Path.cwd() if start is None else Path(start)
    for candidate in (current.resolve(), *current.resolve().parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate repository root')


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
PROCESSED_DATA_PATH = REPO_ROOT / 'data' / 'processed' / 'v1_processed.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

EVALUATION_TABLE_PATH = RESULTS_DIR / 'v1_evaluation_table.csv'
REQUIRED_FEDD_TABLE_PATH = RESULTS_DIR / 'v1_required_fedd_by_seed_mass.csv'
REQUIRED_MSEED_TABLE_PATH = RESULTS_DIR / 'v1_required_mseed_by_growth_assumption.csv'
SAMPLE_SUMMARY_PATH = RESULTS_DIR / 'v1_sample_summary.csv'

print(f'Repository root: {REPO_ROOT}')
print(f'Processed catalogue: {PROCESSED_DATA_PATH}')

In [ ]:
from models import (
    SEED_MODELS,
    apply_mbh_interpretation,
    apply_mstar_agn_contamination,
    available_growth_time_gyr,
    evaluate_seed_model,
    required_fedd_for_seed,
    required_seed_mass_for_growth,
    run_growth_sanity_checks,
)
from scoring import score_model_table


df = pd.read_csv(PROCESSED_DATA_PATH)
if df.empty:
    raise ValueError('Processed v1 catalogue is empty')
if not df['measurement_id'].is_unique:
    raise ValueError('Processed v1 catalogue has duplicate measurement_id values')

growth_checks = run_growth_sanity_checks()
print(f'Loaded {len(df)} processed rows')
print('Growth sanity checks passed:', growth_checks)

In [ ]:
# Keep the v1 interpretation variants from the original notebook.
INTERPRETATION_VARIANTS = {
    'baseline': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.0},
    'mbh_minus_0p3dex': {'mbh_delta_dex': -0.3, 'mstar_agn_fraction': 0.0},
    'mbh_plus_0p3dex': {'mbh_delta_dex': 0.3, 'mstar_agn_fraction': 0.0},
    'mstar_agn_20pct': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.2},
}

# Current v1 growth assumptions used for required seed-mass tables.
GROWTH_CONFIGS = {
    'eddington_eps0p1': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 1.0},
    'subeddington_eps0p1': {'f_edd_avg': 0.3, 'epsilon': 0.1, 'merger_boost': 1.0},
    'supercritical_eps0p05': {'f_edd_avg': 2.0, 'epsilon': 0.05, 'merger_boost': 1.0},
    'merger_boost_x2': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 2.0},
}

# README interpretability thresholds for required f_Edd tables.
SEED_MASS_ASSUMPTIONS = {
    'seed_1e2_msun': 2.0,
    'seed_1e4_msun': 4.0,
    'seed_1e5_msun': 5.0,
}

# Unique epsilon/merger cases for solving required f_Edd. f_Edd itself is the unknown here.
FEDD_REQUIREMENT_CONFIGS = {
    'eps0p1_no_merger_boost': {'epsilon': 0.1, 'merger_boost': 1.0},
    'eps0p05_no_merger_boost': {'epsilon': 0.05, 'merger_boost': 1.0},
    'eps0p1_merger_boost_x2': {'epsilon': 0.1, 'merger_boost': 2.0},
}

Z_SEED_V1 = 30.0
OBJECT_METADATA_COLUMNS = [
    'measurement_id', 'object_id', 'redshift', 'redshift_kind', 'survey',
    'object_class', 'quality_flag', 'source_key', 'source_table',
    'missing_mstar_flag', 'missing_lbol_flag', 'missing_edd_ratio_flag',
    'missing_lensing_flag',
]

In [ ]:
def object_metadata(obj: pd.Series) -> dict[str, object]:
    return {col: obj[col] for col in OBJECT_METADATA_COLUMNS if col in obj.index}


def evaluated_masses(obj: pd.Series, variant: dict[str, float]) -> tuple[float, float, float]:
    log_mbh = float(apply_mbh_interpretation(obj['log_mbh_msun_std'], variant['mbh_delta_dex']))
    log_mstar = float(apply_mstar_agn_contamination(obj['log_mstar_msun_std'], variant['mstar_agn_fraction']))
    return log_mbh, log_mstar, log_mbh - log_mstar


def fedd_label(required_fedd: float) -> str:
    if pd.isna(required_fedd):
        return 'missing'
    if required_fedd <= 1.0:
        return 'eddington_or_below'
    if required_fedd <= 3.0:
        return 'super_eddington'
    return 'extreme'


def seed_label(required_log_mseed: float) -> str:
    if pd.isna(required_log_mseed):
        return 'missing'
    if required_log_mseed <= 2.0:
        return 'light_seed_scale'
    if required_log_mseed <= 4.0:
        return 'intermediate_seed_scale'
    if required_log_mseed <= 6.0:
        return 'heavy_seed_scale'
    return 'above_heavy_seed_scale'


def score_label(score: float) -> str:
    if pd.isna(score):
        return 'missing'
    if score >= 0.8:
        return 'strong'
    if score >= 0.5:
        return 'partial'
    return 'poor'

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for growth_name, growth in GROWTH_CONFIGS.items():
            for seed_model_name, seed_model in SEED_MODELS.items():
                log_mseed_mid = 0.5 * (seed_model.log_mseed_min + seed_model.log_mseed_max)
                required_fedd = float(
                    required_fedd_for_seed(
                        log_mseed=log_mseed_mid,
                        log_mbh_final=log_mbh,
                        epsilon=growth['epsilon'],
                        z_seed=Z_SEED_V1,
                        z_obs=obj['redshift'],
                        merger_boost=growth['merger_boost'],
                    )
                )
                seed_eval = evaluate_seed_model(
                    log_mbh_final=log_mbh,
                    delta_t_gyr=delta_t,
                    model_name=seed_model_name,
                    f_edd_avg=growth['f_edd_avg'],
                    epsilon=growth['epsilon'],
                    merger_boost=growth['merger_boost'],
                )

                rows.append({
                    **base_metadata,
                    'z_seed': Z_SEED_V1,
                    'delta_t_gyr': delta_t,
                    'interpretation_variant': variant_name,
                    'mbh_delta_dex': variant['mbh_delta_dex'],
                    'mstar_agn_fraction': variant['mstar_agn_fraction'],
                    'log_mbh_eval': log_mbh,
                    'log_mstar_eval': log_mstar,
                    'log_mbh_mstar_ratio_eval': log_ratio,
                    'growth_config': growth_name,
                    'f_edd_avg': growth['f_edd_avg'],
                    'epsilon': growth['epsilon'],
                    'merger_boost': growth['merger_boost'],
                    'seed_model': seed_model_name,
                    'seed_assumption_log_mseed_mid': log_mseed_mid,
                    'seed_assumption_mseed_mid_msun': 10 ** log_mseed_mid,
                    'required_fedd': required_fedd,
                    **seed_eval,
                })

evaluation_df = score_model_table(pd.DataFrame(rows))
evaluation_df['required_fedd_label'] = evaluation_df['required_fedd'].map(fedd_label)
evaluation_df['required_mseed_label'] = evaluation_df['required_log_mseed'].map(seed_label)
evaluation_df['feasibility_label'] = evaluation_df['feasibility_score'].map(score_label)
evaluation_df.to_csv(EVALUATION_TABLE_PATH, index=False)

print(f'Saved {len(evaluation_df)} rows: {EVALUATION_TABLE_PATH}')
evaluation_df.head()

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for config_name, config in FEDD_REQUIREMENT_CONFIGS.items():
            for seed_label_name, log_mseed in SEED_MASS_ASSUMPTIONS.items():
                required_fedd = float(
                    required_fedd_for_seed(
                        log_mseed=log_mseed,
                        log_mbh_final=log_mbh,
                        epsilon=config['epsilon'],
                        z_seed=Z_SEED_V1,
                        z_obs=obj['redshift'],
                        merger_boost=config['merger_boost'],
                    )
                )

                rows.append({
                    **base_metadata,
                    'z_seed': Z_SEED_V1,
                    'delta_t_gyr': delta_t,
                    'interpretation_variant': variant_name,
                    'mbh_delta_dex': variant['mbh_delta_dex'],
                    'mstar_agn_fraction': variant['mstar_agn_fraction'],
                    'log_mbh_eval': log_mbh,
                    'log_mstar_eval': log_mstar,
                    'log_mbh_mstar_ratio_eval': log_ratio,
                    'fedd_requirement_config': config_name,
                    'epsilon': config['epsilon'],
                    'merger_boost': config['merger_boost'],
                    'seed_mass_assumption': seed_label_name,
                    'log_mseed_assumption': log_mseed,
                    'mseed_assumption_msun': 10 ** log_mseed,
                    'required_fedd': required_fedd,
                    'required_fedd_label': fedd_label(required_fedd),
                })

required_fedd_df = pd.DataFrame(rows)
required_fedd_df.to_csv(REQUIRED_FEDD_TABLE_PATH, index=False)

print(f'Saved {len(required_fedd_df)} rows: {REQUIRED_FEDD_TABLE_PATH}')
required_fedd_df.head()

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for growth_name, growth in GROWTH_CONFIGS.items():
            required_log_mseed = float(
                required_seed_mass_for_growth(
                    log_mbh_final=log_mbh,
                    f_edd=growth['f_edd_avg'],
                    epsilon=growth['epsilon'],
                    z_seed=Z_SEED_V1,
                    z_obs=obj['redshift'],
                    merger_boost=growth['merger_boost'],
                )
            )

            rows.append({
                **base_metadata,
                'z_seed': Z_SEED_V1,
                'delta_t_gyr': delta_t,
                'interpretation_variant': variant_name,
                'mbh_delta_dex': variant['mbh_delta_dex'],
                'mstar_agn_fraction': variant['mstar_agn_fraction'],
                'log_mbh_eval': log_mbh,
                'log_mstar_eval': log_mstar,
                'log_mbh_mstar_ratio_eval': log_ratio,
                'growth_config': growth_name,
                'f_edd_avg': growth['f_edd_avg'],
                'epsilon': growth['epsilon'],
                'merger_boost': growth['merger_boost'],
                'required_log_mseed': required_log_mseed,
                'required_mseed_msun': 10 ** required_log_mseed,
                'required_mseed_label': seed_label(required_log_mseed),
            })

required_mseed_df = pd.DataFrame(rows)
required_mseed_df.to_csv(REQUIRED_MSEED_TABLE_PATH, index=False)

print(f'Saved {len(required_mseed_df)} rows: {REQUIRED_MSEED_TABLE_PATH}')
required_mseed_df.head()

In [ ]:
sample_summary_df = (
    evaluation_df
    .groupby(['interpretation_variant', 'growth_config', 'seed_model'], as_index=False)
    .agg(
        n_rows=('measurement_id', 'size'),
        n_objects=('measurement_id', 'nunique'),
        median_required_fedd=('required_fedd', 'median'),
        max_required_fedd=('required_fedd', 'max'),
        median_required_log_mseed=('required_log_mseed', 'median'),
        min_required_log_mseed=('required_log_mseed', 'min'),
        max_required_log_mseed=('required_log_mseed', 'max'),
        feasible_seed_model_fraction=('is_feasible', 'mean'),
        mean_feasibility_score=('feasibility_score', 'mean'),
        median_feasibility_score=('feasibility_score', 'median'),
    )
    .sort_values(['interpretation_variant', 'growth_config', 'seed_model'])
)
sample_summary_df.to_csv(SAMPLE_SUMMARY_PATH, index=False)

print(f'Saved {len(sample_summary_df)} rows: {SAMPLE_SUMMARY_PATH}')
sample_summary_df.head()

In [ ]:
expected_counts = {
    'evaluation': len(df) * len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS) * len(SEED_MODELS),
    'required_fedd': len(df) * len(INTERPRETATION_VARIANTS) * len(FEDD_REQUIREMENT_CONFIGS) * len(SEED_MASS_ASSUMPTIONS),
    'required_mseed': len(df) * len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS),
    'sample_summary': len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS) * len(SEED_MODELS),
}

actual_counts = {
    'evaluation': len(evaluation_df),
    'required_fedd': len(required_fedd_df),
    'required_mseed': len(required_mseed_df),
    'sample_summary': len(sample_summary_df),
}
if expected_counts != actual_counts:
    raise AssertionError(f'Unexpected row counts: expected {expected_counts}, got {actual_counts}')

duplicate_checks = {
    'evaluation': evaluation_df.duplicated(['measurement_id', 'interpretation_variant', 'growth_config', 'seed_model']).sum(),
    'required_fedd': required_fedd_df.duplicated(['measurement_id', 'interpretation_variant', 'fedd_requirement_config', 'seed_mass_assumption']).sum(),
    'required_mseed': required_mseed_df.duplicated(['measurement_id', 'interpretation_variant', 'growth_config']).sum(),
    'sample_summary': sample_summary_df.duplicated(['interpretation_variant', 'growth_config', 'seed_model']).sum(),
}
if any(count != 0 for count in duplicate_checks.values()):
    raise AssertionError(f'Accidental duplicate rows found: {duplicate_checks}')

for path in [EVALUATION_TABLE_PATH, REQUIRED_FEDD_TABLE_PATH, REQUIRED_MSEED_TABLE_PATH, SAMPLE_SUMMARY_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Expected row counts:', expected_counts)
print('Duplicate checks:', duplicate_checks)
print('Generated CSVs:')
for path in [EVALUATION_TABLE_PATH, REQUIRED_FEDD_TABLE_PATH, REQUIRED_MSEED_TABLE_PATH, SAMPLE_SUMMARY_PATH]:
    print(f'  {path.relative_to(REPO_ROOT)}')